In this script, we demonstrate how to initialize and change values in TFP.

In [2]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
tfd = tfp.distributions
tfb = tfp.bijectors


Let's set up a Beta prior/posterior so that we have a clearly constrained posterior support of (0, 1).

In [3]:
default_fit = tfp.experimental.vi.build_factored_surrogate_posterior(
    event_shape=(),
    bijector=tfb.Sigmoid(),
)

init_fit = tfp.experimental.vi.build_factored_surrogate_posterior(
    event_shape=(),
    bijector=tfb.Sigmoid(),
    initial_parameters={
        'loc': tf.Variable(1.0),
        'scale': 0.8,
    }
)

To see the details on what parameters to submit to the `build_factored_surrogate_posterior` function, you can either call `help(tfp.experimental.vi.build_factored_surrogate_posterior)` or go to the documentation [website](https://www.tensorflow.org/probability/api_docs/python/tfp/experimental/vi/build_factored_surrogate_posterior) (last visited by the authors of this notebook in June 2026). According to the documentation, the default scale value is always 0.01 and the default mean value is randomly drawn from a unit Normal distribution (you can verify this by running the following code block multiple times to watch the initial mean change).

## What are the default values and what initialization do we have control over?

We will first demonstrate the default mean and scale values, then show how to change them (and what we can and cannot change), and then demonstrate whether the initialization values initialize in the constrained space or the unconstrained gaussian space.

In [12]:
default_fit = tfp.experimental.vi.build_factored_surrogate_posterior(
    event_shape=(),
    bijector=tfb.Sigmoid(),
)
default_initial_values = default_fit.distribution.loc.numpy(), default_fit.distribution.scale.numpy()
init_initial_values = init_fit.distribution.loc.numpy(), init_fit.distribution.scale.numpy()
print(f"The initialized values by default are {default_initial_values[0]}, {default_initial_values[1]}")
print(f"The initialized values when we set loc=1.0 and scale=0.8 are {init_initial_values[0]}, {init_initial_values[1]}")

The initialized values by default are 1.5456840991973877, 0.009999997913837433
The initialized values when we set loc=1.0 and scale=0.8 are 1.0, 0.8000000715255737


Note: (on line 62-64 of _default_parameter_init_fn in trainable.py of tfp) the scale values get cast to float32 which cannot represent 0.01 or 0.8 exactly, so we check that what they return is the same as the floating point representation. Note that they are first being stored as float64, so we have to cast to float64 and then to float32 to check this precision. This is the reason for the long decimal value for the 0.8 initialization above.

In [4]:
# empirically observe the mean and std of the samples
init_samples = init_fit.sample(100_000)
default_samples = default_fit.sample(100_000)
print(f"Default fit (constrained) empirical mean and std: {np.mean(default_samples.numpy())}, {np.std(default_samples.numpy())}")
print(f"Init fit (constrained) empirical mean and std: {np.mean(init_samples.numpy())}, {np.std(init_samples.numpy())}")

Default fit (constrained) empirical mean and std: 0.48628485202789307, 0.002491940511390567
Init fit (constrained) empirical mean and std: 0.7071442008018494, 0.15043577551841736


It seems that the values we observe empirically are different from the initialized values, so let's transform the initialized values and see if they match the empirical values. (we use measure transport, i.e. we draw samples from the unconstrained distribution, transform the samples, and then take the empirical mean and std of the transformed samples)

In [5]:
# sample from the initialized values and transform to constrained space to find empirical constrained mean and std from
# initial values
init_constrained_samples = tfb.Sigmoid().forward(
    tfd.Normal(loc=init_fit.distribution.loc, scale=init_fit.distribution.scale).sample(100_000)
)
default_constrained_samples = tfb.Sigmoid().forward(
    tfd.Normal(loc=default_fit.distribution.loc, scale=default_fit.distribution.scale).sample(100_000)
)
print(f"Default fit empirical constrained mean and std from initial values: {np.mean(default_constrained_samples.numpy())}, {np.std(default_constrained_samples.numpy())}")
print(f"Init fit empirical constrained mean and std from initial values: {np.mean(init_constrained_samples.numpy())}, {np.std(init_constrained_samples.numpy())}")

Default fit empirical constrained mean and std from initial values: 0.4862838387489319, 0.002490274142473936
Init fit empirical constrained mean and std from initial values: 0.7078702449798584, 0.14973874390125275


Since these values match the constrained mean and std above, we conclude that TFP's initialization control changes the underlying unconstrained gaussian distribution in ADVI.